# Dispatch Cost Reduction — 2025

Compares actual 2025 dispatch spend (from `transfers.csv`) against two simulation scenarios:

| Scenario | Description |
|----------|-------------|
| **Actual** | Real 2025 dispatches from `transfers.csv` |
| **Baseline sim** | Proactive overflow prevention + actual non-overflow transfers (repair routing, demand rebalancing, remarketing) replayed from `transfers.csv` at full trucks |
| **Optimised sim** | Proactive overflow prevention + cross-compound repair batching (`RepairBatcher`) — replaces actual repair routing with optimised full-truckload consolidation |

The key hypothesis: **repair cars accumulate at non-repair compounds and trigger expensive emergency overflow dispatches (2× surcharge). Proactive, full-truckload repair routing eliminates this cycle.**

In [ ]:
import os, sys
os.chdir('..')
sys.path.insert(0, '.')

import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

In [ ]:
# ── Load data ──────────────────────────────────────────────────────────────────
car_events  = pd.read_csv('data/car_events.csv',              parse_dates=['event_date'])
transfers   = pd.read_csv('data/transfers.csv',               parse_dates=['transfer_date', 'scheduled_date'])
compounds   = pd.read_csv('data/compounds.csv',               parse_dates=['opened_date'])
demand      = pd.read_csv('data/demand_forecast_inputs.csv',  parse_dates=['month'])

In [ ]:
# ── Preprocessing ──────────────────────────────────────────────────────────────
from src.preprocessing import flag_repair_cars, impute_repair_estimate, add_region_to_car_events

car_events = flag_repair_cars(car_events, transfers)
car_events = impute_repair_estimate(car_events)
car_events = add_region_to_car_events(car_events, compounds)

In [ ]:
# ── Haversine distance matrix ──────────────────────────────────────────────────
def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlam = np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlam/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

dist_matrix = {}
for _, r1 in compounds.iterrows():
    for _, r2 in compounds.iterrows():
        if r1.compound_id != r2.compound_id:
            dist_matrix[(r1.compound_id, r2.compound_id)] = haversine(
                r1.latitude, r1.longitude, r2.latitude, r2.longitude
            )

print(f'Distance matrix: {len(dist_matrix)} routes')

In [ ]:
# ── Initial simulation state at 2025-01-06 ─────────────────────────────────────
from src.simulation import SimulationState, HorizonSimulationRunner
from src.simulation.stock import (
    apply_repair_fifo, calibrate_initial_stock,
    build_stock_series, assess_compound_status,
)

SIM_START = pd.Timestamp('2025-01-06')
SIM_END   = pd.Timestamp('2025-12-28')

# FIFO repair scheduling on pre-simulation data (required by RepairStateTracker)
ev_pre = car_events[car_events['event_date'] < SIM_START].copy()
ev_train_sched = apply_repair_fifo(ev_pre, compounds)

# Initial stock (anchored via overflow history)
initial_stock_map = calibrate_initial_stock(car_events, transfers, compounds)
stock_series      = build_stock_series(
    car_events, compounds, initial_stock_map,
    start_date=car_events['event_date'].min(),
    end_date=SIM_START,
)
stock_jan  = stock_series.loc[SIM_START] if SIM_START in stock_series.index else stock_series.iloc[-1]
stock_jan1 = stock_jan.to_dict()

# Repair queue (waiting_repair) and repair load (in_repair) at sim start
compound_status = assess_compound_status(
    ev_train_sched, compounds,
    as_of_date=SIM_START,
    stock={cpd: int(round(v)) for cpd, v in stock_jan1.items()},
)
repair_queue_init = compound_status.set_index('compound_id')['waiting_repair'].fillna(0).astype(int).to_dict()
repair_load_init  = compound_status.set_index('compound_id')['in_repair'].fillna(0).astype(int).to_dict()
stock_init        = {cpd: int(round(v)) for cpd, v in stock_jan1.items()}

initial_state = SimulationState(
    stock        = stock_init,
    repair_queue = repair_queue_init,
    repair_load  = repair_load_init,
    week         = SIM_START,
)
print('Initial stock (Jan 6):', sum(stock_init.values()), 'cars across', len(stock_init), 'compounds')

---
## Section 1 — Actual 2025 baseline (transfers.csv)

In [ ]:
from src.config import TRUCK_CAPACITY_CARS, EMERGENCY_SURCHARGE

t25 = transfers[transfers['transfer_date'].dt.year == 2025].copy()

# cost_eur is per-car (truck_cost / cars_on_truck).
# Summing all rows = total truck spend (each car's share × cars_on_truck = truck cost).
actual_total_cost  = t25['cost_eur'].sum()
actual_total_cars  = len(t25)

# Unique trucks: one row per car, group by truck_id
truck_agg = t25.groupby('truck_id').agg(
    cars_per_truck=('cars_per_truck', 'first'),
    fill_rate     =('truck_fill_rate', 'first'),
    truck_cost    =('cost_eur', lambda x: x.sum()),  # sum of per-car costs = truck cost
    reason        =('reason', 'first'),
    distance_km   =('distance_km', 'first'),
).reset_index()
actual_total_trucks = len(truck_agg)

print(f'2025 Actual Dispatch Spend')
print(f'  Total cost   : €{actual_total_cost:,.0f}')
print(f'  Total trucks : {actual_total_trucks}')
print(f'  Total cars   : {actual_total_cars}')
print(f'  Avg fill rate: {truck_agg.fill_rate.mean():.1%}')

In [ ]:
# ── Cost breakdown by reason ───────────────────────────────────────────────────
reason_summary = (
    t25.groupby('reason')
    .agg(
        cars        =('car_id', 'count'),
        total_cost  =('cost_eur', 'sum'),
        avg_fill    =('truck_fill_rate', 'mean'),
    )
    .assign(pct_cost=lambda df: df['total_cost'] / df['total_cost'].sum() * 100)
    .sort_values('total_cost', ascending=False)
    .round({'total_cost': 0, 'avg_fill': 3, 'pct_cost': 1})
)
print('Cost by reason:')
print(reason_summary.to_string())

In [ ]:
# ── Partial-load waste quantification ─────────────────────────────────────────
# Waste = cost of empty truck slots.
# A truck costs the same regardless of fill, so empty slots = wasted spend.
# Waste per truck = truck_cost × (1 − fill_rate) = cost of unused capacity.
from src.config import TRUCK_BASE_FEE_EUR, TRUCK_PER_KM_EUR

truck_agg['waste_per_truck']    = truck_agg['truck_cost'] * (1 - truck_agg['fill_rate'])
partial_waste                   = truck_agg['waste_per_truck'].sum()
partial_trucks                  = (truck_agg['fill_rate'] < 1.0).sum()

# Minimum trucks needed to move the same cars at 100% fill
min_trucks_needed = int(np.ceil(actual_total_cars / TRUCK_CAPACITY_CARS))
trucks_saved_at_100pct = actual_total_trucks - min_trucks_needed

print(f'Partial-load waste:')
print(f'  Trucks below 100% fill : {partial_trucks} / {actual_total_trucks}')
print(f'  Cost of empty slots    : €{partial_waste:,.0f}  ({partial_waste / actual_total_cost:.1%} of total spend)')
print(f'  Min trucks at 100% fill: {min_trucks_needed}  (vs {actual_total_trucks} actual = {trucks_saved_at_100pct} excess trucks)')

In [ ]:
# ── Emergency surcharge cost ───────────────────────────────────────────────────
# lead_time_days == 1 → emergency dispatch (2× rate)
em_rows = t25[t25['lead_time_days'] <= 1]
em_cost = em_rows['cost_eur'].sum()
em_cost_at_standard = em_cost / EMERGENCY_SURCHARGE
em_surcharge_premium = em_cost - em_cost_at_standard

print(f'Emergency dispatch (lead_time ≤ 1 day):')
print(f'  Cars         : {len(em_rows)}')
print(f'  Actual cost  : €{em_cost:,.0f} (at 2× rate)')
print(f'  Standard cost: €{em_cost_at_standard:,.0f} (at 1× rate)')
print(f'  Surcharge    : €{em_surcharge_premium:,.0f}')

---
## Section 2 — Baseline simulation (overflow prevention + actual non-overflow transfers)

In [ ]:
# All non-overflow transfers: repair_routing + demand_rebalancing + remarketing_routing
non_overflow_transfers = transfers[transfers['reason'] != 'capacity_overflow']

runner_baseline = HorizonSimulationRunner(
    car_events          = car_events,
    demand              = demand,
    compounds           = compounds,
    dist_matrix         = dist_matrix,
    ev_train_sched      = ev_train_sched,
    horizon_weeks       = 4,
    urgency_window      = 2,
    use_repair_batching = False,
    actual_transfers    = non_overflow_transfers,
)
runner_baseline.run(initial_state, SIM_START, SIM_END)
print('\n── Baseline ──')
print(runner_baseline.summary())

---
## Section 3 — Optimised simulation (overflow prevention + repair batching + actual rebalancing & remarketing)

In [ ]:
# Rebalancing + remarketing transfers are replayed as-is (not optimised).
# Only repair_routing is replaced by RepairBatcher.
passthrough_transfers = transfers[
    transfers['reason'].isin(['demand_rebalancing', 'remarketing_routing'])
]

runner_opt = HorizonSimulationRunner(
    car_events          = car_events,
    demand              = demand,
    compounds           = compounds,
    dist_matrix         = dist_matrix,
    ev_train_sched      = ev_train_sched,
    horizon_weeks       = 4,
    urgency_window      = 2,
    use_repair_batching = True,
    actual_transfers    = passthrough_transfers,
)
runner_opt.run(initial_state, SIM_START, SIM_END)
print('\n── Optimised ──')
print(runner_opt.summary())

---
## Section 4 — Cost comparison

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────
def _sim_metrics(runner):
    wdf = runner.week_df()
    cdf = runner.cost_df()
    hdf = runner.handover_df()
    fill = (cdf['cars'] / (cdf['trucks'] * TRUCK_CAPACITY_CARS)).mean() if not cdf.empty else 0
    shortfall_wks = int((hdf.groupby('week')['shortfall'].sum() > 0).sum()) if not hdf.empty else 0
    return {
        'total_cost_eur'       : wdf['cost_eur'].sum() if not wdf.empty else 0,
        'total_trucks'         : int(wdf['trucks'].sum()) if not wdf.empty else 0,
        'avg_fill_rate'        : fill,
        'overflow_cpd_weeks'   : int(wdf['overflow_compounds'].sum()) if not wdf.empty else 0,
        'handover_shortfall_wks': shortfall_wks,
    }

metrics = pd.DataFrame({
    'Actual 2025'       : {
        'total_cost_eur'        : actual_total_cost,
        'total_trucks'          : actual_total_trucks,
        'avg_fill_rate'         : truck_agg['fill_rate'].mean(),
        'overflow_cpd_weeks'    : None,
        'handover_shortfall_wks': None,
    },
    'Baseline sim'      : _sim_metrics(runner_baseline),
    'Optimised sim'     : _sim_metrics(runner_opt),
}).T

metrics['saving_vs_actual_eur'] = actual_total_cost - metrics['total_cost_eur']
metrics['saving_vs_actual_pct'] = metrics['saving_vs_actual_eur'] / actual_total_cost * 100

print(metrics[['total_cost_eur','total_trucks','avg_fill_rate',
               'overflow_cpd_weeks','handover_shortfall_wks',
               'saving_vs_actual_eur','saving_vs_actual_pct']].to_string())

In [ ]:
# ── Cost breakdown by reason — optimised run ───────────────────────────────────
cdf_opt = runner_opt.cost_df()
if not cdf_opt.empty:
    reason_opt = (
        cdf_opt.groupby('reason')
        .agg(trucks=('trucks','sum'), cars=('cars','sum'), cost=('cost_eur','sum'))
        .assign(avg_fill=lambda df: df['cars'] / (df['trucks'] * TRUCK_CAPACITY_CARS))
        .sort_values('cost', ascending=False)
        .round({'cost': 0, 'avg_fill': 3})
    )
    print('Optimised run — cost by reason:')
    print(reason_opt.to_string())

In [ ]:
# ── Spending breakdown helpers ────────────────────────────────────────────────

TRANSFER_TYPES = [
    "Emergency overflow",
    "Planned overflow",
    "Repair routing",
    "Demand rebalancing",
    "Remarketing routing",
]

# actual transfers.csv uses capacity_overflow for both emergency and planned;
# distinguish by lead_time_days (<= 1 = booked same/next day = emergency)
_ACTUAL_MASKS = {
    "Emergency overflow" : lambda df: (df["reason"] == "capacity_overflow") & (df["lead_time_days"] <= 1),
    "Planned overflow"   : lambda df: (df["reason"] == "capacity_overflow") & (df["lead_time_days"] > 1),
    "Repair routing"     : lambda df:  df["reason"] == "repair_routing",
    "Demand rebalancing" : lambda df:  df["reason"] == "demand_rebalancing",
    "Remarketing routing": lambda df:  df["reason"] == "remarketing_routing",
}

_SIM_REASONS = {
    "Emergency overflow" : "emergency_overflow",
    "Planned overflow"   : "horizon_overflow",
    "Repair routing"     : "repair_routing",
    "Demand rebalancing" : "demand_rebalancing",
    "Remarketing routing": "remarketing_routing",
}

def actual_stats_by_type(transfers_df):
    """
    Extract (total_cost_eur, n_cars) per transfer type from transfers.csv.
    One row per car; cost_eur is per-car, so sum = total truck spend.
    """
    return {
        label: (transfers_df[mask(transfers_df)]["cost_eur"].sum(),
                int(mask(transfers_df).sum()))
        for label, mask in _ACTUAL_MASKS.items()
    }

def sim_stats_by_type(cost_df):
    """
    Extract (total_cost_eur, n_cars) per transfer type from runner.cost_df().
    One row per dispatch; cars column holds car count for that dispatch.
    """
    return {
        label: (
            cost_df[cost_df["reason"] == reason]["cost_eur"].sum(),
            int(cost_df[cost_df["reason"] == reason]["cars"].sum()),
        )
        for label, reason in _SIM_REASONS.items()
    }

def build_cost_breakdown(actual_stats, baseline_stats, optimised_stats):
    """
    Build a DataFrame comparing cost and cost-per-car across three scenarios
    for each transfer type, plus a Total row.

    Parameters
    ----------
    actual_stats, baseline_stats, optimised_stats : dict
        Output of actual_stats_by_type() or sim_stats_by_type().

    Returns
    -------
    pd.DataFrame
    """
    rows = {}
    for k in TRANSFER_TYPES:
        ca, na = actual_stats[k]
        cb, nb = baseline_stats[k]
        co, no = optimised_stats[k]
        rows[k] = {
            "Act cost (€)"  : ca,
            "Act €/car"     : ca / na if na else float("nan"),
            "Base cost (€)" : cb,
            "Base €/car"    : cb / nb if nb else float("nan"),
            "Opt cost (€)"  : co,
            "Opt €/car"     : co / no if no else float("nan"),
            "Δ B→O (€)"     : co - cb,
            "Δ B→O (%)"     : (co - cb) / cb * 100 if cb else float("nan"),
        }
    df = pd.DataFrame(rows).T.loc[TRANSFER_TYPES]

    def _totals(stats):
        return sum(v[0] for v in stats.values()), sum(v[1] for v in stats.values())

    tot_ca, tot_na = _totals(actual_stats)
    tot_cb, tot_nb = _totals(baseline_stats)
    tot_co, tot_no = _totals(optimised_stats)
    totals = pd.DataFrame([{
        "Act cost (€)"  : tot_ca,
        "Act €/car"     : tot_ca / tot_na if tot_na else float("nan"),
        "Base cost (€)" : tot_cb,
        "Base €/car"    : tot_cb / tot_nb if tot_nb else float("nan"),
        "Opt cost (€)"  : tot_co,
        "Opt €/car"     : tot_co / tot_no if tot_no else float("nan"),
        "Δ B→O (€)"     : tot_co - tot_cb,
        "Δ B→O (%)"     : (tot_co - tot_cb) / tot_cb * 100 if tot_cb else float("nan"),
    }], index=["Total"])
    return pd.concat([df, totals])

def format_cost_breakdown(df):
    """Format a cost breakdown DataFrame for display."""
    def _fmt(val, col):
        if pd.isna(val):
            return "-"
        if col.endswith("(%)"):
            return f"{val:+.1f}%"
        if "€/car" in col:
            return f"€{val:.2f}"
        return f"€{val:,.0f}"

    fmt = df.copy()
    for col in fmt.columns:
        fmt[col] = fmt[col].map(lambda x, c=col: _fmt(x, c))
    return fmt

# ── Build and display ─────────────────────────────────────────────────────────
actual_s   = actual_stats_by_type(t25)
baseline_s = sim_stats_by_type(runner_baseline.cost_df())
opt_s      = sim_stats_by_type(runner_opt.cost_df())


### adding a manual fix 
t25_rema_rebaln = t25[t25['reason'].isin(['remarketing_routing', 'demand_rebalancing'])]
base_without = runner_baseline.cost_df()[~runner_baseline.cost_df()['reason'].isin(['remarketing_routing', 'demand_rebalancing'])]
opt_without = runner_opt.cost_df()[~runner_opt.cost_df()['reason'].isin(['remarketing_routing', 'demand_rebalancing'])]
optimised_new = pd.concat([opt_without, t25_rema_rebaln], axis = 0)
base_new = pd.concat([base_without , t25_rema_rebaln], axis = 0)

bd = build_cost_breakdown(actual_s, sim_stats_by_type(base_new ), sim_stats_by_type(optimised_new))

print("Spending breakdown by transfer type (cost + cost-per-car)")
print(format_cost_breakdown(bd).to_string())

In [ ]:
# ── Graph 1: Cost per car — actual vs optimised ──────────────────────────────
# ── Graph 2: Total optimised costs by flow type ──────────────────────────────

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

### very manual fix 
opt_s = sim_stats_by_type(optimised_new)

FLOW_LABELS = [
    "Emergency overflow",
    "Planned overflow",
    "Repair routing",
    "Demand rebalancing",
    "Remarketing routing",
]

act_per_car = [
    actual_s[k][0] / actual_s[k][1] if actual_s[k][1] else 0
    for k in FLOW_LABELS
]
opt_per_car = [
    opt_s[k][0] / opt_s[k][1] if opt_s[k][1] else 0
    for k in FLOW_LABELS
]
### manual fix
opt_per_car[3] = act_per_car[3]
opt_per_car[4] = act_per_car[4]

opt_total = [opt_s[k][0] for k in FLOW_LABELS]
act_total = [actual_s[k][0] for k in FLOW_LABELS]

x = np.arange(len(FLOW_LABELS))
w = 0.35

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle("Optimised simulation — cost analysis by flow type", fontsize=13, fontweight="bold")

# ── Graph 1: Cost per car ─────────────────────────────────────────────────────
ax1 = axes[0]
bars_a = ax1.bar(x - w/2, act_per_car, w, label="Actual cost", color="#4C9BE8")
bars_o = ax1.bar(x + w/2, opt_per_car, w, label="Optimised cost", color="#3AAA7A")

for bar in bars_a:
    h = bar.get_height()
    if h > 0:
        ax1.text(bar.get_x() + bar.get_width()/2, h + 1, f"€{h:.0f}",
                 ha="center", va="bottom", fontsize=8)
for bar in bars_o:
    h = bar.get_height()
    if h > 0:
        ax1.text(bar.get_x() + bar.get_width()/2, h + 1, f"€{h:.0f}",
                 ha="center", va="bottom", fontsize=8)

ax1.set_title("Cost per car by flow type")
ax1.set_ylabel("€ per car")
ax1.set_xticks(x)
ax1.set_xticklabels([l.replace(" ", "\n") for l in FLOW_LABELS], fontsize=9)
ax1.set_xlabel("Flow type")
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"€{v:.0f}"))
ax1.legend()
ax1.grid(axis="y", linestyle="--", alpha=0.4)

# ── Graph 2: Total optimised costs ───────────────────────────────────────────

ax2 = axes[1]
bars_tt = ax2.bar(x - w/2, act_total, w, color="#4C9BE8", edgecolor="white", label="Actual cost")
bars_t  = ax2.bar(x + w/2, opt_total, w, color="#3AAA7A", edgecolor="white", label="Optimised cost")

for bar in bars_t:
    h = bar.get_height()
    if h > 0:
        ax2.text(bar.get_x() + bar.get_width()/2, h + 200, f"€{h:,.0f}",
                 ha="center", va="bottom", fontsize=8)
        
for bar in bars_tt:
    h = bar.get_height()
    if h > 0:
        ax2.text(bar.get_x() + bar.get_width()/2, h + 200, f"€{h:,.0f}",
                 ha="center", va="bottom", fontsize=8)
        

ax2.set_title("Total optimised cost by flow type")
ax2.set_ylabel("Total cost (€)")
ax2.set_xticks(x)
ax2.set_xticklabels([l.replace(" ", "\n") for l in FLOW_LABELS], fontsize=9)
ax2.set_xlabel("Flow type")
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"€{v:,.0f}"))
ax2.grid(axis="y", linestyle="--", alpha=0.4)

# Total annotation
total_opt = sum(opt_total)
ax2.axhline(0, color="none")
ax2.text(0.98, 0.97, f"Total: €{total_opt:,.0f}",
         transform=ax2.transAxes, ha="right", va="top",
         fontsize=10, fontweight="bold",
         bbox=dict(boxstyle="round,pad=0.3", facecolor="#3AAA7A", alpha=0.15))

plt.tight_layout()
plt.show()

In [ ]:
# ── Weekly cost chart: baseline vs optimised ───────────────────────────────────
wdf_b = runner_baseline.week_df().set_index('week')['cost_eur'].rename('Baseline')
wdf_o = runner_opt.week_df().set_index('week')['cost_eur'].rename('Optimised')

fig, ax = plt.subplots(figsize=(14, 4))
wdf_b.plot(ax=ax, label='Baseline (overflow only)', color='steelblue', linewidth=1.5)
wdf_o.plot(ax=ax, label='Optimised (+ repair batching)', color='seagreen', linewidth=1.5)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'€{x:,.0f}'))
ax.set_title('Weekly dispatch cost: baseline vs optimised')
ax.set_xlabel('')
ax.legend()
plt.tight_layout()
plt.show()

# ── Fill rate distribution: baseline vs optimised ──────────────────────────────
cdf_b = runner_baseline.cost_df()
if not cdf_b.empty and not cdf_opt.empty:
    fill_b = cdf_b['cars'] / (cdf_b['trucks'] * TRUCK_CAPACITY_CARS)
    fill_o = cdf_opt['cars'] / (cdf_opt['trucks'] * TRUCK_CAPACITY_CARS)
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(fill_b, bins=20, alpha=0.6, label='Baseline', color='steelblue')
    ax.hist(fill_o, bins=20, alpha=0.6, label='Optimised', color='seagreen')
    ax.set_xlabel('Truck fill rate')
    ax.set_title('Fill rate distribution (per dispatch action)')
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Final savings summary ──────────────────────────────────────────────────────
opt_cost  = runner_opt.week_df()['cost_eur'].sum()
base_cost = runner_baseline.week_df()['cost_eur'].sum()

# Upper bound: cost at 100% fill for same actual moves
avg_cost_per_truck  = actual_total_cost / actual_total_trucks
upper_bound_saving  = actual_total_cost - min_trucks_needed * avg_cost_per_truck

print('=== Cost reduction summary ===')
print(f'Actual 2025 spend              : €{actual_total_cost:>10,.0f}  ({actual_total_trucks} trucks, {truck_agg.fill_rate.mean():.0%} avg fill)')
print()
print(f'--- Actual waste drivers ---')
print(f'  Empty-slot waste (partial loads): €{partial_waste:>8,.0f}  ({partial_waste/actual_total_cost:.1%})')
print(f'  Emergency surcharge premium     : €{em_surcharge_premium:>8,.0f}  ({em_surcharge_premium/actual_total_cost:.1%})')
print(f'  Upper bound saving (100% fill)  : €{upper_bound_saving:>8,.0f}  ({upper_bound_saving/actual_total_cost:.1%})')
print()
print(f'--- Simulation comparison ---')
print(f'  Baseline sim (overflow + actual transfers)  : €{base_cost:>8,.0f}')
print(f'  Optimised sim (overflow + repair batching)  : €{opt_cost:>8,.0f}  ({(opt_cost - actual_total_cost)/actual_total_cost:+.1%} vs actual)')
print(f'  Saving: optimised vs baseline               : €{base_cost - opt_cost:>8,.0f}  ({(base_cost - opt_cost)/base_cost:.1%} reduction)')
print()
print(f'Achievable saving vs actual: €{actual_total_cost - opt_cost:,.0f} ({(actual_total_cost - opt_cost)/actual_total_cost:.1%})')

---
## Section 5 — Handover shortage metrics

How many customers couldn't receive their car each week, broken down by compound.
No optimisation applied — this is a diagnostic view of service level under each scenario.

> **Note**: shortfalls here reflect the simulation model's limitations as much as real logistics gaps.
> Cars dispatched to repair compounds show as unavailable until repair completes; the simulation
> does not model repair completion for newly dispatched cars, so shortfalls at repair compounds
> are partly structural (FIFO bay occupancy, not resolvable by additional trucks).

In [ ]:
# ── Aggregate shortfall stats per scenario ─────────────────────────────────────
def _shortfall_stats(runner, label):
    hdf = runner.handover_df()
    if hdf.empty:
        return {}
    weekly = hdf.groupby('week').agg(
        demand   =('demand', 'sum'),
        shortfall=('shortfall', 'sum'),
        available=('available', 'sum'),
    ).reset_index()
    total_demand    = weekly['demand'].sum()
    total_shortfall = weekly['shortfall'].sum()
    peak_shortfall  = weekly['shortfall'].max()
    peak_week       = weekly.loc[weekly['shortfall'].idxmax(), 'week']
    shortfall_weeks = (weekly['shortfall'] > 0).sum()
    service_rate    = 1 - total_shortfall / total_demand if total_demand > 0 else 1.0
    # Max cumulative queue (carry-forward backlog at any single week)
    max_queue       = hdf.groupby('week')['queue_backlog'].sum().max()
    return {
        'scenario'        : label,
        'total_demand'    : int(total_demand),
        'total_shortfall' : int(total_shortfall),
        'service_rate'    : service_rate,
        'shortfall_weeks' : int(shortfall_weeks),
        'peak_wk_shortfall': int(peak_shortfall),
        'peak_week'       : peak_week.date(),
        'max_queue_backlog': int(max_queue),
    }

stats = pd.DataFrame([
    _shortfall_stats(runner_baseline, 'Baseline sim'),
    _shortfall_stats(runner_opt,      'Optimised sim'),
]).set_index('scenario')

print('Handover shortage summary:')
print(stats.to_string())

In [ ]:
# ── Weekly shortfall chart: baseline vs optimised ─────────────────────────────
hdf_b = runner_baseline.handover_df().groupby('week')['shortfall'].sum().rename('Baseline')
hdf_o = runner_opt.handover_df().groupby('week')['shortfall'].sum().rename('Optimised')

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

# Top: weekly shortfall
ax = axes[0]
hdf_b.plot(ax=ax, label='Baseline (overflow only)', color='steelblue', linewidth=1.5)
hdf_o.plot(ax=ax, label='Optimised (+ repair batching)', color='seagreen', linewidth=1.5)
ax.set_ylabel('Cars short (new demand)')
ax.set_title('Weekly handover shortfall: cars that could not be delivered')
ax.legend()
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

# Bottom: cumulative queue carry-forward
queue_b = runner_baseline.handover_df().groupby('week')['queue_backlog'].sum().rename('Baseline')
queue_o = runner_opt.handover_df().groupby('week')['queue_backlog'].sum().rename('Optimised')
ax2 = axes[1]
queue_b.plot(ax=ax2, label='Baseline (overflow only)', color='steelblue', linewidth=1.5)
queue_o.plot(ax=ax2, label='Optimised (+ repair batching)', color='seagreen', linewidth=1.5)
ax2.set_ylabel('Cumulative queue (cars owed)')
ax2.set_title('Cumulative handover queue: carry-forward backlog from prior weeks')
ax2.legend()
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

plt.tight_layout()
plt.show()

In [ ]:
# ── Per-compound shortfall breakdown (optimised run) ──────────────────────────
# Which compounds drive the most shortfall? Helps identify structural vs dispatch issues.
hdf_o_full = runner_opt.handover_df()
cpd_stats = (
    hdf_o_full.groupby('compound_id')
    .agg(
        total_demand   =('demand', 'sum'),
        total_shortfall=('shortfall', 'sum'),
        weeks_short    =('shortfall', lambda x: (x > 0).sum()),
        avg_available  =('available', 'mean'),
    )
    .assign(
        service_rate=lambda df: 1 - df['total_shortfall'] / df['total_demand'].replace(0, np.nan)
    )
    .sort_values('total_shortfall', ascending=False)
    .round({'service_rate': 3, 'avg_available': 1})
)
# Join repair_capable flag
cpd_stats = cpd_stats.join(
    compounds.set_index('compound_id')[['region', 'repair_capable']]
)
print('Per-compound shortage (optimised run, full year):')
print(cpd_stats[cpd_stats['total_shortfall'] > 0].to_string())